# CNN

CNN，全称是 Convolutional Neural Network，即卷积神经网络。它最早在图像任务中被大量使用，到现在仍然是计算机视觉中非常基础、非常重要的一类结构。

如果说 Linear 层会把输入的所有特征一次性连接到输出特征，那么 CNN 的核心想法就是：

> 不需要一开始就看完整张图，而是先用一个小窗口在局部区域里提取特征，再把这些局部特征组合起来。

对于图像来说，相邻像素之间通常有很强的关系。例如一条边缘、一个角点、一小块纹理，往往只需要观察局部区域就能发现。CNN 正是利用了这种局部相关性。

## CNN 是什么

在 2D CNN 中，输入通常写成：

$$
X \in \mathbb{R}^{B \times C_{in} \times H \times W}
$$

其中：

- $B$ 是 batch size。
- $C_{in}$ 是输入通道数，例如 RGB 图像中 $C_{in}=3$。
- $H$ 和 $W$ 分别是输入特征图的高度和宽度。

卷积层中的权重，也就是卷积核，通常写成：

$$
W \in \mathbb{R}^{C_{out} \times C_{in} \times K_h \times K_w}
$$

其中：

- $C_{out}$ 是输出通道数，也可以理解为卷积核组的数量。
- $C_{in}$ 表示每个输出通道都会同时看所有输入通道。
- $K_h$ 和 $K_w$ 是卷积核在空间维度上的大小。

下面这张图展示了 2D convolution 的主要计算过程：

![2D Convolution in CNNs](../figs/cnn.png)

需要注意的是，深度学习框架里通常说的 convolution，实际计算时一般不会翻转 kernel，更准确地说是 cross-correlation。不过在 CNN 语境下，大家通常仍然把它称为 convolution。

## CNN 的计算过程

以输出中的一个位置 $Y[b, oc, h, w]$ 为例，它的计算可以分成三步。

第一步，对于某一个输出通道 $oc$，取出对应的卷积核：

$$
W[oc] \in \mathbb{R}^{C_{in} \times K_h \times K_w}
$$

第二步，在输入的每个 channel 上取一个局部窗口，与卷积核对应位置做逐元素相乘并求和。

第三步，把所有输入 channel 的结果加起来，再加上这个输出通道自己的 bias：

$$
Y[b, oc, h, w]
= \sum_{ic=0}^{C_{in}-1} \sum_{i=0}^{K_h-1} \sum_{j=0}^{K_w-1}
X_{pad}[b, ic, h \cdot stride + i, w \cdot stride + j]
\cdot W[oc, ic, i, j] + bias[oc]
$$

直观理解就是：

- 一个卷积核负责生成一个输出 channel。
- 每个输出 channel 都会看全部输入 channel。
- 卷积核在图像空间上滑动，每滑到一个位置，就计算一个输出位置。
- bias 是按输出 channel 添加的，所以每个 $oc$ 只有一个 bias。

本 notebook 下面的 NumPy 实现也是按照这个逻辑写的：先遍历 batch，再遍历 output channel，再遍历 input channel，最后遍历输出特征图上的每个空间位置。

## CNN 特性

CNN 之所以适合图像任务，主要来自几个特点。

1. 局部连接

卷积核每次只看一个局部窗口，而不是直接连接整张图。这符合图像的局部相关性：边缘、纹理、角点这些低层特征通常都来自局部区域。

2. 参数共享

同一个卷积核会在整张图上滑动。也就是说，不同空间位置使用的是同一组参数。

这带来一个很重要的好处：如果某个卷积核学会了检测“竖直边缘”，那么无论这个边缘出现在图片左上角还是右下角，它都可以被同一个卷积核检测到。

3. 平移等变性

卷积本身更准确地说具有平移等变性（translation equivariance）：如果输入图像平移了一点，那么输出特征图也会相应平移。

有时我们也会说 CNN 具有一定的平移不变性（translation invariance），但这通常来自后续的 pooling、stride、全局平均池化等操作，而不是单独一个卷积层直接带来的。

4. 更少的参数量

相比 Linear 层直接连接整张图，卷积层只学习一个小窗口里的参数，并在空间位置上共享，因此参数量会小很多。

## 计算 output 的 shape

假设输入 shape 是：

$$
[B, C_{in}, H, W]
$$

卷积核 shape 是：

$$
[C_{out}, C_{in}, K_h, K_w]
$$

padding 为 $P$，stride 为 $S$。如果这里先假设高度和宽度方向使用相同的 padding 和 stride，那么输出高度和宽度为：

$$
H_{out} = \left\lfloor \frac{H + 2P - K_h}{S} \right\rfloor + 1
$$

$$
W_{out} = \left\lfloor \frac{W + 2P - K_w}{S} \right\rfloor + 1
$$

所以输出 shape 是：

$$
Y \in \mathbb{R}^{B \times C_{out} \times H_{out} \times W_{out}}
$$


因此：

$$
H_{out} = \left\lfloor \frac{5 + 2 \times 1 - 3}{1} \right\rfloor + 1 = 5
$$

$$
W_{out} = \left\lfloor \frac{5 + 2 \times 1 - 3}{1} \right\rfloor + 1 = 5
$$

最终输出 shape 为：

$$
[1, 2, 5, 5]
$$


## CNN 的参数量

假设输入 shape 是：

$$
[B, C_{in}, H, W]
$$

卷积层的参数只和输入通道数、输出通道数、卷积核大小有关，和输入图片的 $H,W$ 没有直接关系。

如果卷积核大小是 $K_h \times K_w$，那么 weight 参数量为：

$$
C_{out} \times C_{in} \times K_h \times K_w
$$

如果使用 bias，每个输出 channel 有一个 bias，因此 bias 参数量为：

$$
C_{out}
$$

总参数量为：

$$
C_{out} \times C_{in} \times K_h \times K_w + C_{out}
$$

以当前例子为例：

$$
C_{in}=3, \quad C_{out}=2, \quad K_h=K_w=3
$$

所以参数量为：

$$
2 \times 3 \times 3 \times 3 + 2 = 56
$$

如果把 $5 \times 5 \times 3$ 的输入直接用 Linear 层映射到 $5 \times 5 \times 2$ 的输出，那么参数量会大很多，而且还失去了图像中的局部结构先验。这也是 CNN 在图像任务中非常有效的原因之一。


In [2]:
# Conv2d implementation using NumPy
import numpy as np

class Conv2d:
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, bias=True):
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.bias = bias

        # Initialize weights and bias
        # shape of weights: (out_channels, in_channels, kernel_size, kernel_size)
        self.weights = np.random.randn(out_channels, in_channels, kernel_size, kernel_size) * 0.01
        if bias:
            # for bias, we have one bias per output channel
            self.biases = np.random.randn(out_channels) * 0.01
        else:
            self.biases = None
    
    def forward(self, x):
        # x shape: (batch_size, in_channels, height, width)
        batch_size, _, height, width = x.shape
        
        # Calculate output dimensions
        out_height = (height + 2 * self.padding - self.kernel_size) // self.stride + 1
        out_width = (width + 2 * self.padding - self.kernel_size) // self.stride + 1    

        # Initialize output tensor
        out = np.zeros((batch_size, self.out_channels, out_height, out_width))

        # Apply padding to the input
        if self.padding > 0:
            x_padded = np.pad(x, ((0, 0), (0, 0), (self.padding, self.padding), (self.padding, self.padding)), mode='constant')
        else:
            x_padded = x

        # Perform convolution
        for b in range(batch_size): # for each image in the batch
            for oc in range(self.out_channels): # for each kernel/output channel
                for ic in range(self.in_channels): # for each input channel
                    for i in range(out_height):
                        for j in range(out_width):
                            # Calculate the start and end indices for the current slice
                            h_start = i * self.stride
                            h_end = h_start + self.kernel_size
                            w_start = j * self.stride
                            w_end = w_start + self.kernel_size

                            # Perform element-wise multiplication and sum
                            out[b, oc, i, j] += np.sum(x_padded[b, ic, h_start:h_end, w_start:w_end] * self.weights[oc, ic])

                # Add bias if applicable
                if self.bias:
                    out[b, oc] += self.biases[oc]
        
        # Return the output tensor
        return out

In [3]:
# Example usage
x = np.random.randn(1, 3, 5, 5)  # Example input: batch size 1, 3 channels, 5x5 image
conv = Conv2d(in_channels=3, out_channels=2, kernel_size=3, stride=1, padding=1)
output = conv.forward(x)
print("Input shape:", x.shape)
print("Output shape:", output.shape)

Input shape: (1, 3, 5, 5)
Output shape: (1, 2, 5, 5)
